# N-ATLaS Health QA: live demo

Loads `NCAIR1/N-ATLaS`, answers a few questions, then loads the published adapter `Emar7/N-ATLaS-Health-QA-LoRA-Powered-by-Awarri` on top and answers the same questions again.

Needs a GPU, internet access, and a secret named `HF_TOKEN`.

In [ ]:
!pip install -q "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33"
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise SystemExit("No HF_TOKEN secret attached to this notebook (Add-ons > Secrets): " + type(e).__name__)

## 1. Load base N-ATLaS

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE = "NCAIR1/N-ATLaS"
ADAPTER = "Emar7/N-ATLaS-Health-QA-LoRA-Powered-by-Awarri"

tok = AutoTokenizer.from_pretrained(BASE)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map={"": 0},
                                             torch_dtype=torch.float16)
model.eval()
print("Loaded base model:", BASE)

## 2. Prepare the questions

In [ ]:
import json, re
import pandas as pd
from huggingface_hub import hf_hub_download

LETTERS = "ABCDEFGH"
LETTER_IDS = [tok.encode(ch, add_special_tokens=False)[0] for ch in LETTERS]
SYSTEM = ("You are a careful medical assistant answering multiple-choice medical exam questions. "
          "Reply with the letter of the single best option.")

def build_prompt(question, options):
    opts = "\n".join(f"{LETTERS[i]}. {t}" for i, t in enumerate(options))
    user = f"Question:\n{question.strip()}\n\nOptions:\n{opts}\n\nAnswer with the letter of the single best option."
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    return user, tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False, date_string="11 Oct 2026")

@torch.no_grad()
def pick_letter(m, prompt, n_options):
    # Same scoring as the benchmark: the option letter with the highest next-token probability.
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(m.device)
    logits = m(**enc).logits[0, -1, :].float()
    probs = torch.softmax(logits[LETTER_IDS[:n_options]], dim=0)
    return LETTERS[int(torch.argmax(probs))], [round(float(p), 2) for p in probs]

@torch.no_grad()
def free_text(m, prompt, max_new_tokens):
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(m.device)
    out = m.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# The first five benchmark questions, in file order, from the published predictions file.
saved = pd.read_csv(hf_hub_download(ADAPTER, "benchmark_500_predictions.csv")).head(5)
data = pd.read_csv(hf_hub_download("intronhealth/afrimedqa_v2", "afri_med_qa_15k_v2.4_phase_2_15275.csv",
                                   repo_type="dataset")).set_index("sample_id")

def order(k):
    m = re.search(r"(\d+)", str(k))
    return int(m.group(1)) if m else 999

bench = []
for _, s in saved.iterrows():
    r = data.loc[s["id"]]
    q = r["question_clean"] if isinstance(r["question_clean"], str) and r["question_clean"].strip() else r["question"]
    d = json.loads(r["answer_options"])
    options = [str(v).strip() for k, v in sorted(d.items(), key=lambda kv: order(kv[0]))
               if v is not None and str(v).strip() and str(v).strip().lower() not in ("nan", "none", "n/a")]
    user, prompt = build_prompt(q, options)
    bench.append({"user": user, "prompt": prompt, "n": len(options), "gold": s["answer_letter"],
                  "saved_base": s["base_pred"], "saved_adapter": s["finetuned_pred"], "specialty": s["specialty"]})

MALARIA_Q = ("A 4-year-old child in a malaria-endemic area presents with fever, pallor and a palpable spleen. "
             "Which investigation should be done first?")
MALARIA_OPTS = ["Chest X-ray", "Blood film for malaria parasites", "Lumbar puncture", "Urine culture"]
malaria_user, malaria_prompt = build_prompt(MALARIA_Q, MALARIA_OPTS)
print("Ready:", len(bench), "benchmark questions and 1 extra question.")

## 3. Base N-ATLaS answers

In [ ]:
print("=" * 78)
print("BASE N-ATLaS (no adapter)")
print("=" * 78)
print(malaria_user)
base_malaria = pick_letter(model, malaria_prompt, len(MALARIA_OPTS))
print("\nBase N-ATLaS picks:", base_malaria[0], "| probability of each option:", base_malaria[1])
print("Base N-ATLaS free-text answer:", free_text(model, malaria_prompt, 60))
base_live = [pick_letter(model, b["prompt"], b["n"]) for b in bench]

## 4. Load the adapter on top of N-ATLaS

In [ ]:
model = PeftModel.from_pretrained(model, ADAPTER)   # the adapter loads on top of N-ATLaS
model.eval()
print("Loaded adapter on top of", model.peft_config["default"].base_model_name_or_path, ":", ADAPTER)

## 5. N-ATLaS with the adapter answers the same questions

In [ ]:
print("=" * 78)
print("N-ATLaS + HEALTH QA ADAPTER")
print("=" * 78)
print(malaria_user)
tuned_malaria = pick_letter(model, malaria_prompt, len(MALARIA_OPTS))
print("\nAdapter picks:", tuned_malaria[0], "| probability of each option:", tuned_malaria[1])
print("Adapter free-text answer:", free_text(model, malaria_prompt, 8))
tuned_live = [pick_letter(model, b["prompt"], b["n"]) for b in bench]

## 6. Side by side

In [ ]:
print("One benchmark question in full (number 2 of the 500):\n")
b = bench[1]
print(b["user"])
print("\nCorrect answer:", b["gold"])
print("Base N-ATLaS answered:", base_live[1][0], "| probabilities:", base_live[1][1])
print("With the adapter:     ", tuned_live[1][0], "| probabilities:", tuned_live[1][1])

rows = []
for i, b in enumerate(bench):
    rows.append({"question": i + 1, "specialty": b["specialty"], "correct": b["gold"],
                 "base (live)": base_live[i][0], "adapter (live)": tuned_live[i][0],
                 "base (published)": b["saved_base"], "adapter (published)": b["saved_adapter"]})
table = pd.DataFrame(rows)
print("\nFirst five benchmark questions, in file order. 'published' is the answer in benchmark_500_predictions.csv.\n")
print(table.to_string(index=False))
same_base = int((table["base (live)"] == table["base (published)"]).sum())
same_tuned = int((table["adapter (live)"] == table["adapter (published)"]).sum())
print(f"\nLive answers matching the published file: base {same_base}/5, adapter {same_tuned}/5.")
print("Full benchmark, 500 questions: base N-ATLaS 252 correct (50.4%), with the adapter 303 correct (60.6%).")
print("Research model. English only. Not a medical device. Not for diagnosis or treatment.")